# POD and Data Treatment

In [29]:
import gdown
import torch

import numpy as np

from scipy.linalg import svd

from joblib import Parallel, delayed

# from podcnf.DataGenerationStokes import ADR, mesh, Vh

from dlroms import num2p

ImportError: cannot import name 'num2p' from 'dlroms' (/home/andreaviolante1501/podcnf/.venv/lib/python3.12/site-packages/dlroms/__init__.py)

## Load Data

In [16]:
filename = "stokes_data_6400.pt"
gdown.download(id = "1_E-gNMU9aMmWXHqm63JspI9i4kWy-wd1", quiet=True, output = filename)
loaded_data = torch.load(filename)

In [23]:
n_samples = loaded_data['u'].shape[0]
print(n_samples)

6400


In [9]:
# Extract the tensors
u = loaded_data['u']
eps = loaded_data['eps'].squeeze(1)
theta = loaded_data['theta'].squeeze(1)
mu = loaded_data['mu']

# Verify the shapes
print(f"u shape: {u.shape}")
print(f"mu shape: {mu.shape}")
print(f"eps shape: {eps.shape}")
print(f"theta shape: {theta.shape}")

u shape: torch.Size([6400, 2763])
mu shape: torch.Size([6400, 3])
eps shape: torch.Size([6400, 1])
theta shape: torch.Size([6400, 1])


In [ ]:
k = 1
indices = np.random.randint(0,6399,k)
plot_stokes_solution(indices, loaded_data, Vh)

Variability visualization

In [10]:
nnnn = 4
c1, c2, c3 = np.repeat(10 * np.random.rand(3, 1) - 5, nnnn, -1)
eps = 0.0025 * np.random.rand(nnnn) + 0.0025
theta = np.pi/2 * np.random.rand(nnnn) - np.pi/4

In [14]:
def solve_single_instance(i, eps_val, theta_val, c1_val, c2_val, c3_val):
    uu = ADR(eps_val, theta_val, c1_val, c2_val, c3_val)
    return uu

results_ex = np.array([
    solve_single_instance(i, eps[i], theta[i], c1[i], c2[i], c3[i])
    for i in range(nnnn)
])

NameError: name 'ADR' is not defined

In [15]:
u_ex.shape

NameError: name 'u_ex' is not defined

In [ ]:
mu_ex = np.stack([c1, c2, c3], axis=1)
dummy_data = {
    'u': torch.tensor(u_ex, dtype=torch.float32),
    'mu': torch.tensor(mu_ex, dtype=torch.float32),
    'eps': torch.tensor(eps, dtype=torch.float32).unsqueeze(1),
    'theta': torch.tensor(theta, dtype=torch.float32).unsqueeze(1)
}

indices = np.array([0, 1, 2, 3], dtype=int)
plot_stokes_solution(indices, dummy_data, Vh)

## SVD

To train our generative model of course it is not possbile to use all the dimensions (2763), for this we represent our solution in a Low-dimensional space. So then what the NF will learn will be the probability distribution of c|c1,c2,c3. Where c are the coefficients given by the SVD obtained with projection matrix V. This solves the computation part. After that will be sufficient to re-project our coefficients into the high-dimensional space to obtaing for instance graphical representation and so on.

For the SVD I've used N_train to compute the SVD and then I've evaluated the performances on the validation set

In [24]:
ntrain = int(n_samples * 0.75)
nval = int(ntrain + n_samples * 0.2)
X, s, _ = svd(u[:ntrain].T, full_matrices = False)

In [25]:
s_array = np.array(s)
s_percentage = s_array / np.sum(s_array, keepdims=True)

Here the cumulative variance given 20 basis as an example

In [26]:
n_basis = 20
cum_s = np.cumsum(s_percentage[:n_basis])
cum_s

array([0.70949376, 0.75922406, 0.78894776, 0.81607324, 0.8388008 ,
       0.8580993 , 0.8715228 , 0.88263935, 0.8919373 , 0.9003091 ,
       0.906639  , 0.9119584 , 0.91675156, 0.92116475, 0.9255071 ,
       0.92937803, 0.93298686, 0.9363062 , 0.9393362 , 0.94232506],
      dtype=float32)

The projection error comptuted as follow:
```
V = X[:, :n_basis]
errors = np.linalg.norm(u[ntrain:nval, :] - u[ntrain:nval, :] @ V @ V.T, axis = 1)/np.linalg.norm(u[ntrain:nval], axis = 1)
```
Relative error 1.05% for 20 basis


In [30]:
V = X[:, :n_basis]
errors = np.linalg.norm(u[ntrain:nval, :] - u[ntrain:nval, :] @ V @ V.T, axis = 1)/np.linalg.norm(u[ntrain:nval], axis = 1)

print("Basis projection error: %s." % num2p(errors.mean()))

/tmp/ipykernel_3103/3939196801.py:2: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  errors = np.linalg.norm(u[ntrain:nval, :] - u[ntrain:nval, :] @ V @ V.T, axis = 1)/np.linalg.norm(u[ntrain:nval], axis = 1)


NameError: name 'num2p' is not defined

In [35]:
c = u @ V

/tmp/ipykernel_3103/530000075.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  c = u @ V


In the residual plot we could observe a uniformly spread random cloud for each cylinder. Each plot represent the relative error defined in this way:
```
residuals = u - u_rec
u_norms = np.linalg.norm(u, axis=1)
error_norms = np.linalg.norm(residuals, axis=1)
relative_errors = np.divide(error_norms, u_norms, out=np.zeros_like(error_norms), where=u_norms!=0)
```
Basically we measure the distance between the true u and the reconstructed one, we take the norm and divided it by u_norms.
We don't see any strange behaviour so we can assume that there is no bias threw certain specific configuration.

For what concerns the Mean Absolute Reconstruction Error for 20 basis: $|u_{true} - u_{rec}|$
We could see that the error is centerd around the cylinder.

In [31]:
mean_field = analyze_stokes_residuals(u, X[:, :n_basis], mu)

NameError: name 'analyze_stokes_residuals' is not defined

In [ ]:
analyze_bases_variation_stokes(u, mu, n_bases_list=range(1, 41))

## Data Treatment

In [37]:
BATCH_SIZE = 64

n_train = int(n_samples * 0.75)
n_val = int(n_train + n_samples * 0.2)
n_test = int(n_samples * 0.05)

shuffle = True
drop_last = False

# norm_scaler = None # No normalization
norm_scaler = True # StandardScaler()
# norm_scaler = False # MinMaxScaler()

print(f"n_train: {n_train}")
print(f"n_val: {n_val}")
print(f"n_test: {n_test}")

n_train: 4800
n_val: 6080
n_test: 320


In [38]:
mu.shape, c.shape

(torch.Size([6400, 3]), torch.Size([6400, 20]))

In [39]:
from podcnf.Loader import LoadData

if norm_scaler == None:
    train_loader, val_loader, test_loader = LoadData(mu, c,
                                                     n_train, n_val,
                                                     BATCH_SIZE,
                                                     norm_scaler,
                                                     shuffle, drop_last)

else:
    train_loader, val_loader, test_loader, mu_scaler, c_scaler = LoadData(mu, c,
                                                                          n_train, n_val,
                                                                          BATCH_SIZE,
                                                                          norm_scaler,
                                                                          drop_last)

In [40]:
print(f"Mean - std (Train): {train_loader.dataset.data.mean()} - {train_loader.dataset.data.std()}")
print(f"Mean - std (Val): {val_loader.dataset.data.mean()} - {val_loader.dataset.data.std()}")
print(f"Mean - std (Test): {test_loader.dataset.data.mean()} - {test_loader.dataset.data.std()}")

Mean - std (Train): 8.292820319333316e-10 - 1.000004529953003
Mean - std (Val): 0.0012280025985091925 - 0.9968515038490295
Mean - std (Test): -0.008430480025708675 - 0.9979554414749146


In [41]:
dim_x = mu.shape[1]
dim_y = c.shape[1]

print(f"Dim parameter space: {dim_x}")
print(f"Dim data space: {dim_y}")

Dim parameter space: 3
Dim data space: 20


In [42]:
print(f"Number of workers: {train_loader.num_workers}")
print(f"Train shape: {train_loader.dataset.data.shape}")
print(f"Val shape: {val_loader.dataset.data.shape}")
print(f"Test shape: {test_loader.dataset.data.shape}")

Number of workers: 2
Train shape: torch.Size([4800, 23])
Val shape: torch.Size([1280, 23])
Test shape: torch.Size([320, 23])


## Saving data for training and anlysis

In [45]:
# ==========================================
# 1. SETUP DATALOADER E NORMALIZZAZIONE
# ==========================================
import os
import pickle
import numpy as np

# Importiamo la tua magica funzione
from podcnf.Loader import LoadData

os.makedirs('../models/scalers', exist_ok=True)

# 1. Carichiamo i dati di partenza e i coefficienti ridotti
mu = np.load('../data/raw/mu_data.npy')
c = np.load('../data/processed/c_processed.npy')

n_samples = mu.shape[0]
n_train = int(n_samples * 0.75)
n_val = int(n_samples * 0.20)

BATCH_SIZE = 64

print("Creazione DataLoader e Normalizzazione in corso...")
# 2. Chiamiamo LoadData: divide i dati, normalizza e crea i DataLoader!
train_loader, val_loader, test_loader, mu_scaler, c_scaler = LoadData(
    mu, c, 
    n_train=n_train, 
    n_val=n_val, 
    BATCH_SIZE=BATCH_SIZE, 
    norm_scaler=True, 
    shuffle=True, 
    drop_last=False
)

# 3. Salviamo gli Scaler in modo sicuro (così l'addestramento è slegato)
with open('../models/scalers/mu_scaler.pkl', 'wb') as f:
    pickle.dump(mu_scaler, f)
with open('../models/scalers/c_scaler.pkl', 'wb') as f:
    pickle.dump(c_scaler, f)

print(f"✅ DataLoader pronti! (Workers: {train_loader.num_workers})")
print(f"Shape del batch di train: {next(iter(train_loader)).shape}")
print("✅ Scaler salvati in '../models/scalers/'")

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/mu_data.npy'